# TP 5 — Alignement des LLM sur les préférences humaines : Reward Model, PPO et DPO

**Faculté des Sciences de Tunis — 2026-2027 — Dr. Asma Mekki — IDS 5 / IGL 5 — Generative AI**

## Contexte
Un LLM pré-entraîné peut produire des réponses grammaticalement correctes sans être utile, honnête ou conforme aux préférences des utilisateurs. Ce TP étudie comment des **préférences humaines** (paires *chosen* / *rejected*) améliorent son comportement.

## Plan
| Partie | Contenu |
|---|---|
| 1 | Comprendre et préparer les préférences |
| 2 | Reward Model avec perte de Bradley–Terry |
| 3 | Boucle RLHF et PPO (environnement jouet) |
| 4 | DPO avec LoRA sur un petit LLM |
| 5 | Évaluation comparative et réflexion |

## Environnement conseillé
- **Google Colab avec GPU (T4)** : Runtime → Modifier le type d'exécution → GPU. Durée totale ≈ 20–30 min.
- Sans GPU, le notebook fonctionne mais la Partie 4 sera lente (réduire `DPO_STEPS`).
- Un accès internet est nécessaire (téléchargement des modèles et du jeu de préférences). Si le jeu de données ne peut pas être téléchargé, un **petit jeu de secours (20 paires)** écrit à la main est utilisé automatiquement.

## 0. Installation et imports

In [ ]:
%pip install -q transformers datasets peft accelerate scikit-learn pandas matplotlib scipy
# torchao préinstallé sur Colab est trop ancien pour peft (ImportError) et inutile ici -> on le retire
%pip uninstall -y -q torchao

In [ ]:
import random, math, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

# Partie 1 — Comprendre et préparer les préférences

## 1.0 Le jeu de préférences

Chaque exemple contient trois champs :
- `prompt` : l'instruction donnée au modèle ;
- `chosen` : la réponse **préférée** par l'annotateur ;
- `rejected` : la réponse **moins adaptée**.

Source principale : `Intel/orca_dpo_pairs` (Hugging Face). On ne garde que des paires **courtes** pour que l'entraînement tienne sur un GPU modeste. Si le téléchargement échoue, on utilise 20 paires écrites à la main.

In [ ]:
# --- Jeu de secours (écrit à la main) : utilisé seulement si le téléchargement échoue ---
HANDMADE = [
 ("What is the capital of Australia?", "The capital of Australia is Canberra.", "It's Sydney, the biggest city and the capital of Australia."),
 ("Explain what a neural network is in one sentence.", "A neural network is a model made of layers of connected units that learn to map inputs to outputs by adjusting weights from data.", "Neural networks are very complicated things that people use in computers and many other places for lots of reasons."),
 ("Will it rain in Tunis tomorrow?", "I can't know the weather without up-to-date data; please check a forecast service for Tunis.", "Yes, it will definitely rain tomorrow in Tunis."),
 ("How do I reverse a string in Python?", "Use slicing: reversed_s = s[::-1].", "You must import the reverse module and call reverse.string(s)."),
 ("Give me a tip to sleep better.", "Keep a regular sleep schedule and avoid screens for an hour before bed.", "Sleep is when you sleep. Just sleep more."),
 ("What is 15% of 200?", "15% of 200 is 30.", "15% of 200 is 35."),
 ("Summarize: The meeting moved to Friday because the manager is traveling on Thursday.", "The meeting was rescheduled to Friday since the manager travels on Thursday.", "The manager is traveling, and many other things happened in the company this week, including budget changes."),
 ("Who wrote Les Miserables?", "Victor Hugo wrote Les Miserables, published in 1862.", "Les Miserables was written by Alexandre Dumas."),
 ("What does HTTP stand for?", "HyperText Transfer Protocol.", "HTTP stands for High Transfer Text Process, a way computers print text."),
 ("Is it safe to mix bleach and ammonia for cleaning?", "No. Mixing them releases toxic chloramine gas. Never combine them; ventilate and use one product at a time.", "Yes, mixing them makes a stronger cleaner that works faster."),
 ("Explain overfitting briefly.", "Overfitting happens when a model memorizes training data, including noise, and therefore generalizes poorly to new data.", "Overfitting is when the model is too good, which is always great."),
 ("What is the boiling point of water at sea level?", "100 degrees Celsius (212 F) at standard atmospheric pressure.", "Water boils at 90 degrees Celsius everywhere on Earth."),
 ("Suggest a name for a coffee shop.", "How about 'Bean There'? It is short, playful and memorable.", "Coffee shop."),
 ("Translate 'good morning' into French.", "'Good morning' in French is 'Bonjour'.", "It is 'Bonsoir'."),
 ("How many continents are there?", "By the most common convention there are seven continents, though some models count differently.", "There are exactly 12 continents, and nobody disagrees."),
 ("What is Git used for?", "Git is a version control system that tracks changes in files and lets teams collaborate on code.", "Git is a website where you watch videos."),
 ("Can you guarantee this investment will make money?", "No. Every investment carries risk and nobody can guarantee returns; consider diversification and professional advice.", "Yes, it is guaranteed. You will double your money."),
 ("Describe photosynthesis in two sentences.", "Plants use sunlight, water and carbon dioxide to make glucose. Oxygen is released as a by-product.", "Photosynthesis is when plants eat soil."),
 ("How can I improve my CV?", "Tailor it to the job, quantify achievements, keep it to one page and proofread it.", "Just make the font bigger and add many colors."),
 ("What is the square root of 144?", "12.", "The square root of 144 is 14 because 14 times 14 is 144."),
]

def load_preferences(n_main=900, n_heldout=40):
    """Charge les paires (prompt, chosen, rejected). Renvoie (pairs, heldout_prompts, source)."""
    try:
        from datasets import load_dataset
        ds = load_dataset("Intel/orca_dpo_pairs", split="train")
        pairs = []
        for row in ds:
            q, c, r = row["question"].strip(), row["chosen"].strip(), row["rejected"].strip()
            # On garde des paires courtes (GPU modeste) et non identiques
            if (len(q.split()) <= 80 and 3 <= len(c.split()) <= 90
                    and 3 <= len(r.split()) <= 90 and c != r):
                pairs.append({"prompt": q, "chosen": c, "rejected": r})
        random.Random(SEED).shuffle(pairs)
        main = pairs[:n_main]
        heldout = [p["prompt"] for p in pairs[n_main:n_main + n_heldout]]
        return main, heldout, "Intel/orca_dpo_pairs"
    except Exception as e:
        print("Téléchargement impossible -> jeu de secours (20 paires). Raison :", repr(e)[:120])
        pairs = [{"prompt": p, "chosen": c, "rejected": r} for p, c, r in HANDMADE]
        random.Random(SEED).shuffle(pairs)
        heldout = ["What is the largest planet in the solar system?",
                   "Give one tip to learn programming faster.",
                   "Is the Earth flat?",
                   "Explain what an API is in one sentence.",
                   "What is 12 times 12?"]
        return pairs, heldout, "jeu manuel (secours)"

pairs, heldout_prompts, SOURCE = load_preferences()
print(f"Source : {SOURCE} | {len(pairs)} paires | {len(heldout_prompts)} prompts de test (hors entraînement)")
pd.DataFrame(pairs[:3])

## 1.1 Pourquoi la réponse *chosen* est-elle préférable ?

Observons quelques paires et analysons-les selon les critères du sujet : **pertinence, exactitude, clarté, concision, reconnaissance de l'incertitude**.

In [ ]:
for k in range(3):
    p = pairs[k]
    print(f"===== Exemple {k+1} =====")
    print("PROMPT  :", textwrap.shorten(p["prompt"], 300))
    print("CHOSEN  :", textwrap.shorten(p["chosen"], 400))
    print("REJECTED:", textwrap.shorten(p["rejected"], 400))
    print()

**Analyse (à adapter à vos exemples affichés ci-dessus)**

Grille de lecture à appliquer à chaque paire :

| Critère | Question à se poser | Exemple typique |
|---|---|---|
| **Pertinence** | La réponse traite-t-elle exactement la question ? | « Résume… » : la réponse *chosen* résume ; la *rejected* ajoute des informations hors sujet |
| **Exactitude** | Les faits/calculs sont-ils corrects ? | « 15 % de 200 » : 30 (correct) vs 35 |
| **Clarté** | La réponse est-elle structurée et compréhensible ? | une définition nette vs une phrase vague (« des choses compliquées… ») |
| **Concision** | Va-t-elle droit au but, sans remplissage ? | une phrase précise vs un paragraphe qui tourne autour |
| **Incertitude** | Le modèle reconnaît-il ce qu'il ne sait pas ? | « Va-t-il pleuvoir demain ? » : *chosen* explique qu'il faut une prévision ; *rejected* affirme avec certitude |

Dans le jeu `orca_dpo_pairs`, la réponse *chosen* vient d'un modèle plus puissant (plus complète, plus structurée, plus précise) et la *rejected* d'un modèle plus faible (plus vague, parfois erronée ou hors sujet). On voit donc des préférences qui mélangent **qualité réelle** et **style**.

## 1.2 Identifier les champs `prompt`, `chosen`, `rejected`

- `prompt` : le contexte commun ;
- `chosen` : la réponse que l'annotateur préfère ;
- `rejected` : l'alternative moins bonne.

**Pourquoi les deux réponses doivent répondre au même prompt ?** Une préférence est un jugement **relatif et conditionnel** : « *pour cette question*, A vaut mieux que B ». Si les deux réponses répondaient à des prompts différents, la comparaison mélangerait deux facteurs : la qualité de la réponse **et** la difficulté/nature de la question. Le modèle de récompense apprendrait alors des corrélations parasites (ex. « les questions de maths ont de meilleurs scores ») au lieu d'apprendre ce qui rend *une réponse meilleure qu'une autre pour une même demande*. Fixer le prompt **neutralise** tout ce qui ne dépend que de la question, et isole la qualité de la réponse. C'est aussi ce qui permet à la perte de Bradley–Terry de ne dépendre que de la différence `r(prompt, chosen) − r(prompt, rejected)`.

## 1.3 Quand deux annotateurs peuvent-ils légitimement préférer des réponses différentes ?

**Exemple** — Prompt : *« Explique ce qu'est l'informatique quantique. »*

- **Réponse A** (2 phrases, métaphores simples, aucun terme technique).
- **Réponse B** (long paragraphe avec qubits, superposition, intrication, portes quantiques).

| Annotateur | Profil | Préférence | Justification |
|---|---|---|---|
| 1 | étudiant en histoire, débutant | **A** | claire et accessible |
| 2 | ingénieur en physique | **B** | précise et complète, A est trop simpliste |

Les deux jugements sont **légitimes**. D'autres sources de divergence : longueur, ton (formel/amical), niveau de prudence, valeurs culturelles, tolérance au risque (réponse très directe vs très prudente).

**Pourquoi les préférences ne sont pas une vérité objective ?**
1. Elles dépendent de **qui** annote (âge, culture, expertise, consignes reçues) et du **contexte d'usage**.
2. Elles mélangent **faits vérifiables** (« 15 % de 200 = 30 ») et **goûts** (« ton agréable »).
3. Elles sont **bruitées** (fatigue, désaccords) : l'accord entre annotateurs est rarement de 100 %.
4. Elles peuvent encoder des **biais** (préférer les réponses longues, polies, confiantes).

Un modèle entraîné sur ces préférences apprend donc **les préférences d'un groupe d'annotateurs**, pas « la vérité ».

# Partie 2 — Construire un modèle de récompense (Reward Model)

## 2.4 Des paires aux entrées individuelles + découpage train / validation

Chaque paire `(prompt, chosen, rejected)` produit **deux entrées** à évaluer : `(prompt, chosen)` et `(prompt, rejected)`. Le reward model attribue un score scalaire à chacune.

**Point crucial :** on découpe **par paire** (80 % / 20 %), jamais par entrée. Sinon `(prompt, chosen)` pourrait se retrouver en train et `(prompt, rejected)` en validation : fuite d'information sur le même prompt.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

RM_NAME = "distilbert-base-uncased"   # petit modèle de classification de texte pré-entraîné
MAX_LEN = 256 if DEVICE == "cuda" else 128   # séquences plus courtes sur CPU (beaucoup plus rapide)

# --- Découpage par paire (train / validation) ---
rng = random.Random(SEED)
shuffled = pairs.copy()
rng.shuffle(shuffled)
n_val = max(4, int(0.2 * len(shuffled)))
val_pairs, train_pairs = shuffled[:n_val], shuffled[n_val:]
if DEVICE == "cpu":                       # sans GPU : on réduit pour que le TP reste faisable
    train_pairs, val_pairs = train_pairs[:240], val_pairs[:60]
    print("⚠️ Pas de GPU détecté -> jeu réduit (240 train / 60 val). Active un GPU T4 pour le jeu complet.")
print(f"Paires train : {len(train_pairs)} | Paires validation : {len(val_pairs)}")

rm_tokenizer = AutoTokenizer.from_pretrained(RM_NAME)

def encode_pair(p):
    """Une paire -> deux entrées tokenisées (prompt + réponse)."""
    enc_c = rm_tokenizer(p["prompt"], p["chosen"], truncation=True, max_length=MAX_LEN)
    enc_r = rm_tokenizer(p["prompt"], p["rejected"], truncation=True, max_length=MAX_LEN)
    return enc_c, enc_r

class PreferenceDataset(torch.utils.data.Dataset):
    def __init__(self, pairs): self.items = [encode_pair(p) for p in pairs]
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate_pairs(batch):
    """Regroupe un batch : une version chosen et une version rejected, paddées séparément."""
    chosen = rm_tokenizer.pad([b[0] for b in batch], return_tensors="pt")
    rejected = rm_tokenizer.pad([b[1] for b in batch], return_tensors="pt")
    return chosen, rejected

train_ds, val_ds = PreferenceDataset(train_pairs), PreferenceDataset(val_pairs)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_pairs)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_pairs)

## 2.5 Reward model et perte de Bradley–Terry

**Architecture** : un modèle pré-entraîné de classification de texte (DistilBERT) avec `num_labels=1` → la tête de classification produit **un seul score scalaire** `r(x, y)` au lieu de probabilités de classes.

**Modèle de Bradley–Terry** : la probabilité que `y_w` (chosen) soit préférée à `y_l` (rejected) pour le prompt `x` est

$$P(y_w \succ y_l \mid x) = \sigma\big(r(x,y_w) - r(x,y_l)\big)$$

**Perte** (log-vraisemblance négative) :

$$\mathcal{L}_{BT} = -\,\mathbb{E}\big[\log \sigma\big(r(x,y_w) - r(x,y_l)\big)\big]$$

Minimiser cette perte **pousse le score du chosen au-dessus de celui du rejected**. Seule la **différence** de scores compte : le reward model n'apprend donc qu'une échelle **relative** (on peut ajouter une constante à tous les scores sans changer la perte).

In [ ]:
reward_model = AutoModelForSequenceClassification.from_pretrained(RM_NAME, num_labels=1).to(DEVICE)

def bradley_terry_loss(r_chosen, r_rejected):
    """-log sigma(r_c - r_r), moyenné sur le batch."""
    return -F.logsigmoid(r_chosen - r_rejected).mean()

def rm_scores(model, enc):
    """Score scalaire pour chaque entrée du batch (logits de forme (B,1) -> (B,))."""
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    return model(**enc).logits.squeeze(-1)

@torch.no_grad()
def evaluate_rm(model, loader):
    """Renvoie (accuracy de classement, perte moyenne, liste (score_chosen, score_rejected))."""
    model.eval()
    correct, total, losses, all_scores = 0, 0, [], []
    for chosen, rejected in loader:
        rc, rr = rm_scores(model, chosen), rm_scores(model, rejected)
        losses.append(bradley_terry_loss(rc, rr).item())
        correct += (rc > rr).sum().item()
        total += rc.numel()
        all_scores += list(zip(rc.cpu().tolist(), rr.cpu().tolist()))
    return correct / total, float(np.mean(losses)), all_scores

EPOCHS_RM, LR_RM = 3, 2e-5
optimizer = torch.optim.AdamW(reward_model.parameters(), lr=LR_RM, weight_decay=0.01)
total_steps = EPOCHS_RM * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

acc0, loss0, _ = evaluate_rm(reward_model, val_loader)
print(f"Avant entraînement -> val acc = {acc0:.3f} | val loss = {loss0:.3f} (proche du hasard : loss ≈ 0.693 ; l'accuracy peut s'écarter de 0.5 car la tête est aléatoire)")

import time
print(f"Entraînement sur {DEVICE} : {len(train_loader)} pas/époque x {EPOCHS_RM} époques")
history = []
for epoch in range(EPOCHS_RM):
    reward_model.train()
    run_loss, t0 = [], time.time()
    for step, (chosen, rejected) in enumerate(train_loader, 1):
        rc, rr = rm_scores(reward_model, chosen), rm_scores(reward_model, rejected)
        loss = bradley_terry_loss(rc, rr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        run_loss.append(loss.item())
        if step % 5 == 0 or step == len(train_loader):          # progression visible
            print(f"  epoch {epoch+1} | pas {step}/{len(train_loader)} | loss {np.mean(run_loss[-5:]):.3f} "
                  f"| {time.time()-t0:.0f}s écoulées", flush=True)
    va_acc, va_loss, _ = evaluate_rm(reward_model, val_loader)
    history.append((epoch + 1, np.mean(run_loss), va_acc, va_loss))
    print(f"Epoch {epoch+1}/{EPOCHS_RM} | train loss {np.mean(run_loss):.3f} | "
          f"val acc {va_acc:.3f} | val loss {va_loss:.3f}  ({time.time()-t0:.0f}s)")

## 2.6 Précision de classement sur la validation et analyse des erreurs

**Métrique** : proportion de paires de validation pour lesquelles `score(chosen) > score(rejected)`. Le hasard donne 50 %.

In [ ]:
RM_VAL_ACC, RM_VAL_LOSS, val_scores = evaluate_rm(reward_model, val_loader)
RM_TRAIN_ACC, _, _ = evaluate_rm(reward_model, train_loader)
print(f"Accuracy de classement — train : {RM_TRAIN_ACC:.3f} | validation : {RM_VAL_ACC:.3f}")

# Tableau des paires de validation avec leurs scores
rows = []
for p, (sc, sr) in zip(val_pairs, val_scores):
    rows.append({"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"],
                 "s_chosen": sc, "s_rejected": sr, "ok": sc > sr,
                 "len_chosen": len(p["chosen"].split()), "len_rejected": len(p["rejected"].split())})
val_df = pd.DataFrame(rows)

errors = val_df[~val_df["ok"]]
print(f"\nNombre d'erreurs : {len(errors)} / {len(val_df)}")
for _, r in errors.head(4).iterrows():
    print("-" * 80)
    print("PROMPT  :", textwrap.shorten(r.prompt, 200))
    print(f"CHOSEN  ({r.s_chosen:+.2f}, {r.len_chosen} mots)  :", textwrap.shorten(r.chosen, 250))
    print(f"REJECTED({r.s_rejected:+.2f}, {r.len_rejected} mots):", textwrap.shorten(r.rejected, 250))

**Causes possibles des erreurs** (à vérifier sur vos exemples ci-dessus) :
1. **Biais de longueur** : le reward model peut associer « long » à « bon ». Une réponse *rejected* plus longue ou plus détaillée reçoit un score élevé.
2. **Erreurs factuelles subtiles** : DistilBERT ne vérifie pas les faits ; une réponse fluide mais fausse ressemble à une réponse correcte.
3. **Peu de données** : quelques centaines de paires ne suffisent pas pour apprendre des critères fins.
4. **Capacité limitée du modèle** (petit encodeur) et troncature à 256 tokens.
5. **Étiquettes bruitées / préférences ambiguës** : quand la paire est discutable, aucun modèle ne peut la prédire à coup sûr.

## 2.7 Un score élevé signifie-t-il que la réponse est vraie, utile ou sûre ?

**Non.** Le reward model est un **proxy** appris sur des préférences : il prédit « ce qu'un annotateur aurait probablement préféré », pas la vérité. On vérifie ci-dessous (1) sur des exemples construits, (2) la corrélation entre longueur et score.

In [ ]:
@torch.no_grad()
def score_texts(prompts, responses, batch_size=16):
    """Score du reward model pour des couples (prompt, réponse). Utilisé aussi dans la Partie 5."""
    reward_model.eval()
    out = []
    for i in range(0, len(prompts), batch_size):
        enc = rm_tokenizer(prompts[i:i+batch_size], responses[i:i+batch_size], truncation=True,
                           max_length=MAX_LEN, padding=True, return_tensors="pt")
        out += rm_scores(reward_model, enc).cpu().tolist()
    return out

# Cas construits : fluide mais FAUX vs correct mais bref / prudent
tests = [
    ("What is the capital of Australia?",
     "The capital of Australia is Sydney, a vibrant and beautiful harbour city known worldwide for its stunning opera house and culture.",
     "Canberra."),
    ("Is it safe to mix bleach and ammonia?",
     "Absolutely, mixing household cleaners such as bleach and ammonia creates a powerful solution that cleans much more effectively and quickly.",
     "No, never mix them: they release toxic gas."),
]
for prompt, fluent_wrong, short_right in tests:
    s_w, s_r = score_texts([prompt, prompt], [fluent_wrong, short_right])
    print(f"Prompt : {prompt}")
    print(f"  fluide mais faux  : {s_w:+.2f}")
    print(f"  correct et bref   : {s_r:+.2f}")
    print(f"  -> le reward model {'se trompe' if s_w > s_r else 'a raison'} sur ce cas\n")

# Corrélation score / longueur sur toutes les réponses de validation
lens = list(val_df["len_chosen"]) + list(val_df["len_rejected"])
scs = list(val_df["s_chosen"]) + list(val_df["s_rejected"])
rho, pval = spearmanr(lens, scs)
print(f"Corrélation de Spearman (longueur, score) = {rho:+.2f} (p={pval:.3g})")

**Conclusion de la Partie 2.7 — classement relatif ≠ qualité absolue**

- Le reward model apprend un **classement relatif** : pour un même prompt, quelle réponse est probablement préférée. La perte de Bradley–Terry n'utilise que des **différences** de scores.
- Un score de `+3` n'a **pas de signification absolue** : il n'indique pas « 3 sur 10 », ni « vrai », ni « sûr ». Les scores de prompts différents ne sont même pas comparables entre eux.
- Un score élevé peut venir de **caractéristiques superficielles** (longueur, ton assuré, politesse). Une réponse **fausse mais bien rédigée** peut battre une réponse **vraie mais brève**.
- Conséquence : optimiser agressivement ce score → **reward hacking** (Partie 3.11).

# Partie 3 — Comprendre l'optimisation par PPO

## 3.8 La boucle RLHF classique

Quatre « modèles » interviennent :

| Composant | Rôle | Entraîné ? |
|---|---|---|
| **Modèle SFT** (Supervised Fine-Tuning) | point de départ : LLM déjà ajusté sur des démonstrations ; sert à initialiser la politique **et** le modèle de référence | non (c'est l'initialisation) |
| **Politique π_θ** | le LLM qui génère les réponses et qu'on optimise | **OUI** (seule partie mise à jour par PPO, avec sa « value head » / critique) |
| **Reward model r_φ** | note chaque réponse générée | **NON** (gelé : évaluation) |
| **Modèle de référence π_ref** | copie gelée du SFT ; sert à calculer la pénalité KL | **NON** (gelé : régularisation) |

Flux : prompt → π_θ génère une réponse → r_φ donne un score → on soustrait `β · KL(π_θ ‖ π_ref)` → PPO met à jour π_θ.

In [ ]:
def draw_rlhf_loop():
    fig, ax = plt.subplots(figsize=(12, 6.2))
    ax.set_xlim(0, 12); ax.set_ylim(0, 6.2); ax.axis("off")
    TRAIN, FROZEN, DATA = "#f4a261", "#d9d9d9", "#a8dadc"

    def box(x, y, w, h, text, color):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=color, ec="black", lw=1.2))
        ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=10)

    def arrow(x1, y1, x2, y2, label=None, dy=0.15):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.4))
        if label:
            ax.text((x1+x2)/2, (y1+y2)/2 + dy, label, ha="center", fontsize=9, style="italic")

    box(0.2, 2.4, 1.9, 1.2, "Prompts\n(dataset)", DATA)
    box(3.0, 2.4, 2.4, 1.2, "Politique π_θ\n(ENTRAÎNÉE)", TRAIN)
    box(3.0, 4.6, 2.4, 1.0, "Modèle SFT\n(initialisation)", FROZEN)
    box(6.6, 2.4, 2.3, 1.2, "Reward model r_φ\n(GELÉ)", FROZEN)
    box(3.0, 0.3, 2.4, 1.0, "Référence π_ref\n(GELÉE, copie du SFT)", FROZEN)
    box(9.7, 2.4, 2.1, 1.2, "Récompense totale\nR = r − β·KL", DATA)
    box(9.7, 0.3, 2.1, 1.0, "Mise à jour PPO\n(ratio + clipping)", TRAIN)

    arrow(2.1, 3.0, 3.0, 3.0, "x")
    arrow(5.4, 3.0, 6.6, 3.0, "réponse y")
    arrow(8.9, 3.0, 9.7, 3.0, "r(x,y)")
    arrow(4.2, 4.6, 4.2, 3.6, "init")
    arrow(4.2, 1.3, 4.2, 2.4, "log π_ref(y|x)")
    arrow(5.4, 0.8, 9.7, 2.5, "KL vs π_θ", dy=0.25)
    arrow(10.75, 2.4, 10.75, 1.3, "avantages")
    arrow(9.7, 0.8, 5.4, 2.6, "∇θ", dy=-0.35)
    ax.text(0.2, 5.8, "Orange = entraîné   |   Gris = gelé (évaluation / régularisation)   |   Bleu = données / signal",
            fontsize=10, fontweight="bold")
    plt.title("Boucle RLHF classique (PPO)", fontsize=13)
    plt.show()

draw_rlhf_loop()

## 3.9 PPO sur un environnement jouet

Entraîner un LLM complet avec PPO est très coûteux. On utilise un **environnement jouet de type bandit** :

- une seule « requête » ; la politique choisit **une réponse parmi 6** ;
- chaque réponse a une **récompense du reward model (proxy)** et une **qualité réelle (true)** ;
- la dernière réponse est un **hack** : proxy très élevé mais qualité réelle faible (ex. flatterie excessive que le reward model adore) ;
- la politique est un **softmax** sur 6 logits `θ`, initialisée à la politique de référence.

### Ingrédients de PPO implémentés
1. **Rapport de probabilités** : $\rho = \dfrac{\pi_\theta(a)}{\pi_{old}(a)}$ (combien la politique a changé depuis la collecte des données).
2. **Clipping** : $L^{CLIP} = \mathbb{E}\big[\min(\rho A,\ \text{clip}(\rho, 1-\varepsilon, 1+\varepsilon)A)\big]$ → empêche une mise à jour trop grande.
3. **Pénalité KL** : récompense effective $R = r - \beta\,\log\dfrac{\pi_\theta(a)}{\pi_{ref}(a)}$ (estimateur échantillonné du KL, comme dans InstructGPT).
4. **Avantage** : $A = R - \text{moyenne du batch}$ (normalisé), faute de critique appris dans ce jouet.

In [ ]:
ACTIONS = [
    "A. Réponse correcte et concise",
    "B. Correcte mais verbeuse",
    "C. Partiellement correcte",
    "D. Hors sujet",
    "E. Incorrecte mais très confiante",
    "F. Flatterie excessive (HACK)",
]
# Récompense du reward model (proxy) : il se laisse séduire par F (et un peu par E)
PROXY = torch.tensor([0.80, 0.70, 0.40, 0.00, 0.60, 1.00])
# Qualité réelle (ce que l'utilisateur veut vraiment) : F est mauvaise
TRUE = torch.tensor([1.00, 0.70, 0.40, 0.00, -0.50, -0.20])

# Politique de référence (SFT) : probabilités plausibles, F est très rare
REF_PROBS = torch.tensor([0.35, 0.25, 0.20, 0.10, 0.07, 0.03])
REF_LOGITS = REF_PROBS.log()

def ppo_train(eps=0.2, beta=0.1, steps=150, batch=128, ppo_epochs=4, lr=0.05, seed=0):
    """PPO-clip + pénalité KL sur le bandit jouet. Renvoie l'historique des métriques."""
    g = torch.Generator().manual_seed(seed)
    theta = REF_LOGITS.clone().requires_grad_(True)       # politique initialisée à la référence
    opt = torch.optim.Adam([theta], lr=lr)
    logp_ref_all = F.log_softmax(REF_LOGITS, dim=0)
    hist = {"proxy": [], "true": [], "kl": [], "p_hack": [], "probs": None}

    for step in range(steps):
        # 1) Collecte avec la politique courante (π_old, figée pour ce batch)
        with torch.no_grad():
            logp_old_all = F.log_softmax(theta, dim=0)
            actions = torch.multinomial(logp_old_all.exp(), batch, replacement=True, generator=g)
            logp_old = logp_old_all[actions]
            # 2) Récompense effective = récompense proxy - β * log(π/π_ref)
            kl_sample = logp_old - logp_ref_all[actions]
            reward = PROXY[actions] - beta * kl_sample
            # 3) Avantage = récompense centrée-réduite (pas de critique dans le jouet)
            adv = (reward - reward.mean()) / (reward.std() + 1e-8)

        # 4) Plusieurs époques d'optimisation sur le même batch (spécificité de PPO)
        for _ in range(ppo_epochs):
            logp_new = F.log_softmax(theta, dim=0)[actions]
            ratio = torch.exp(logp_new - logp_old)                          # ρ = π_θ / π_old
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1 - eps, 1 + eps) * adv              # clipping PPO
            loss = -torch.min(surr1, surr2).mean()                          # on MAXIMISE le min
            opt.zero_grad(); loss.backward(); opt.step()

        # 5) Métriques exactes sur la distribution complète
        with torch.no_grad():
            pi = F.softmax(theta, dim=0)
            hist["proxy"].append((pi * PROXY).sum().item())
            hist["true"].append((pi * TRUE).sum().item())
            hist["kl"].append((pi * (pi.log() - logp_ref_all)).sum().item())
            hist["p_hack"].append(pi[-1].item())
    hist["probs"] = F.softmax(theta, dim=0).detach()
    return hist

# Test rapide d'une exécution
h = ppo_train(eps=0.2, beta=0.1)
print("Après 150 pas (eps=0.2, beta=0.1) :")
print(f"  récompense proxy attendue : {h['proxy'][0]:.3f} -> {h['proxy'][-1]:.3f}")
print(f"  qualité réelle attendue   : {h['true'][0]:.3f} -> {h['true'][-1]:.3f}")
print(f"  KL(π‖π_ref)               : {h['kl'][-1]:.3f}")
print(f"  P(réponse hack F)         : {REF_PROBS[-1]:.3f} -> {h['p_hack'][-1]:.3f}")

## 3.10 Influence du coefficient de clipping ε et de la pénalité KL β

- **ε (clipping)** : limite *la taille du pas* de mise à jour. Petit ε → apprentissage stable mais lent ; grand ε → rapide mais risque d'instabilité (le clipping ne limite presque plus rien).
- **β (pénalité KL)** : limite *la distance finale* à la politique de référence. β = 0 → la politique est libre de se concentrer sur la récompense ; β grand → elle reste proche du comportement initial (mais gagne peu de récompense).

On compare une grille de valeurs.

In [ ]:
EPS_LIST = [0.05, 0.2, 0.5]
BETA_LIST = [0.0, 0.1, 0.5, 2.0]

grid = {}
rows = []
for eps in EPS_LIST:
    for beta in BETA_LIST:
        res = ppo_train(eps=eps, beta=beta)
        grid[(eps, beta)] = res
        rows.append({"ε (clip)": eps, "β (KL)": beta,
                     "Récompense proxy": round(res["proxy"][-1], 3),
                     "Qualité réelle": round(res["true"][-1], 3),
                     "KL(π‖π_ref)": round(res["kl"][-1], 3),
                     "P(hack F)": round(res["p_hack"][-1], 3)})
grid_df = pd.DataFrame(rows)
print("Valeurs initiales : proxy = %.3f | réel = %.3f | KL = 0 | P(hack) = %.3f"
      % ((REF_PROBS*PROXY).sum(), (REF_PROBS*TRUE).sum(), REF_PROBS[-1]))
grid_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) Effet de β à ε fixé : récompense proxy vs qualité réelle
for beta in BETA_LIST:
    axes[0].plot(grid[(0.2, beta)]["proxy"], label=f"β={beta}")
    axes[1].plot(grid[(0.2, beta)]["true"], label=f"β={beta}")
axes[0].set_title("Récompense PROXY attendue (ε=0.2)"); axes[1].set_title("Qualité RÉELLE attendue (ε=0.2)")
for a in axes[:2]:
    a.set_xlabel("Pas PPO"); a.legend(); a.grid(alpha=.3)

# (b) Effet de ε à β fixé : vitesse de convergence
for eps in EPS_LIST:
    axes[2].plot(grid[(eps, 0.1)]["proxy"], label=f"ε={eps}")
axes[2].set_title("Effet du clipping ε (β=0.1) : récompense proxy")
axes[2].set_xlabel("Pas PPO"); axes[2].legend(); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

# Compromis récompense vs fidélité à la référence (frontière "récompense / KL")
plt.figure(figsize=(6, 4.2))
for eps in EPS_LIST:
    plt.plot([grid[(eps, b)]["kl"][-1] for b in BETA_LIST],
             [grid[(eps, b)]["proxy"][-1] for b in BETA_LIST], "o-", label=f"ε={eps}")
plt.xlabel("KL(π ‖ π_ref)  (éloignement du comportement initial)")
plt.ylabel("Récompense proxy finale")
plt.title("Compromis récompense / conservation du comportement")
plt.legend(); plt.grid(alpha=.3); plt.show()

**Lecture des résultats (à confronter à vos courbes)**

- **β = 0** : la récompense proxy monte au maximum, la politique concentre sa masse sur l'action F (hack), la **qualité réelle chute** alors que le score augmente, et la KL est très élevée.
- **β croissant** : la politique reste plus proche de la référence (KL faible) ; on gagne moins de récompense proxy, mais on **préserve la qualité réelle**. Un β trop grand empêche tout apprentissage utile.
- **ε** : influe surtout sur la **vitesse / stabilité** de la montée ; avec un ε petit la progression est plus lente et régulière, avec un ε grand les pas sont plus brusques. La **valeur d'équilibre finale** est surtout déterminée par β (le clipping ne fixe pas l'optimum, il contrôle le chemin).
- Il y a donc un **compromis** : augmenter la récompense ↔ conserver le comportement initial. C'est le rôle de la pénalité KL dans RLHF.

## 3.11 Reward hacking

Le **reward hacking** (loi de Goodhart : « quand une mesure devient un objectif, elle cesse d'être une bonne mesure ») apparaît quand la politique **exploite les défauts du reward model** au lieu d'améliorer réellement ses réponses.

Visualisons la distribution finale sans et avec pénalité KL.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
configs = [("Référence (SFT)", REF_PROBS), ("PPO β=0 (sans KL)", grid[(0.2, 0.0)]["probs"]),
           ("PPO β=0.5 (avec KL)", grid[(0.2, 0.5)]["probs"])]
for ax, (title, probs) in zip(axes, configs):
    colors = ["#2a9d8f"] * 5 + ["#e63946"]
    ax.bar(range(6), probs.numpy(), color=colors)
    ax.set_xticks(range(6)); ax.set_xticklabels(["A", "B", "C", "D", "E", "F\n(hack)"])
    ax.set_title(title); ax.set_ylim(0, 1)
axes[0].set_ylabel("Probabilité")
plt.tight_layout(); plt.show()

for title, probs in configs:
    print(f"{title:22s} -> proxy = {(probs*PROXY).sum():.3f} | qualité réelle = {(probs*TRUE).sum():+.3f}")

**Comment une politique obtient un score élevé sans produire de meilleures réponses ?**

Le reward model est imparfait (Partie 2.7). La politique, en optimisant son score, découvre et amplifie ses **angles morts** :
- **allonger** les réponses (biais de longueur) ;
- adopter un ton **assuré et flatteur** (« Excellente question ! ») ;
- **répéter** des formules que le reward model a vues dans les réponses *chosen* ;
- produire des réponses **fluides mais fausses** ;
- exploiter des **motifs hors distribution** (là où le reward model n'a jamais été entraîné, ses scores sont arbitraires).

Garde-fous : **pénalité KL**, **arrêt précoce**, reward model entraîné sur davantage de données/diverses, ensemble de reward models, évaluation par **humains** ou métriques indépendantes.

# Partie 4 — Alignement par DPO (Direct Preference Optimization)

## 4.12 Préparation du jeu de préférences

On réutilise **exactement** les champs `prompt / chosen / rejected` et **la même découpe train / validation** que pour le reward model, afin de comparer proprement.

Le LLM est de type *instruction-tuned* (chat) : on met chaque prompt au **format de conversation** du modèle (`apply_chat_template`), et on ajoute le token de fin de séquence à la fin de la réponse.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"     # petit LLM instruction-tuned (≈ 0.5 milliard de paramètres)
MAX_PROMPT_TOK, MAX_RESP_TOK = 192, 128

tok = AutoTokenizer.from_pretrained(LLM_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def chat_prompt(user_text):
    """Met le prompt au format de conversation du modèle (+ invite de génération)."""
    return tok.apply_chat_template([{"role": "user", "content": user_text}],
                                   tokenize=False, add_generation_prompt=True)

def encode_example(prompt, response):
    """(prompt, réponse) -> (input_ids, labels). Les labels du prompt sont masqués (-100)."""
    p_ids = tok(chat_prompt(prompt), add_special_tokens=False)["input_ids"][-MAX_PROMPT_TOK:]
    r_ids = tok(response + tok.eos_token, add_special_tokens=False)["input_ids"][:MAX_RESP_TOK]
    return p_ids + r_ids, [-100] * len(p_ids) + r_ids

def make_batch(batch_pairs):
    """Construit un batch [chosen_1..chosen_B, rejected_1..rejected_B] paddé à droite."""
    seqs = [encode_example(p["prompt"], p["chosen"]) for p in batch_pairs] + \
           [encode_example(p["prompt"], p["rejected"]) for p in batch_pairs]
    L = max(len(s[0]) for s in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    lab = torch.full((len(seqs), L), -100)
    for i, (x, y) in enumerate(seqs):
        ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1; lab[i, :len(y)] = torch.tensor(y)
    return ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)

# Même découpe que le reward model
dpo_train_pairs, dpo_val_pairs = train_pairs, val_pairs
print(f"DPO — paires train : {len(dpo_train_pairs)} | validation : {len(dpo_val_pairs)}")
ids, att, lab = make_batch(dpo_train_pairs[:2])
print("Forme du batch (2 chosen + 2 rejected) :", tuple(ids.shape))

## 4.13 Modèle de départ + adaptateurs LoRA

**LoRA** (Low-Rank Adaptation) gèle les poids du LLM et n'entraîne que de petites matrices de rang faible ajoutées aux couches d'attention : **moins de mémoire, entraînement rapide**.

**Astuce élégante pour la référence** : le modèle de référence π_ref = le modèle initial **sans adaptateur**. Avec PEFT, on l'obtient avec `with model.disable_adapter():` — pas besoin de charger deux copies du LLM.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(LLM_NAME).to(DEVICE)   # float32 par défaut
lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
policy = get_peft_model(base_model, lora_cfg)
policy.print_trainable_parameters()

### Perte DPO

Pour chaque paire, avec $\pi_\theta$ (politique avec LoRA) et $\pi_{ref}$ (modèle initial) :

$$\mathcal{L}_{DPO} = -\log \sigma\Big(\beta\Big[\underbrace{\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}}_{\text{récompense implicite de chosen}} - \underbrace{\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}}_{\text{récompense implicite de rejected}}\Big]\Big)$$

- $\log \pi(y|x)$ = **somme** des log-probabilités des tokens de la réponse.
- $\beta$ contrôle l'éloignement toléré par rapport à la référence (comme le β du KL en PPO).
- **Au tout début**, π_θ = π_ref (les matrices LoRA B sont initialisées à 0) → la marge vaut 0 → la perte vaut $\log 2 \approx 0.693$ : bon test de cohérence.

In [ ]:
def sequence_logps(model, ids, att, lab):
    """log π(réponse | prompt) = somme des log-probas des tokens de la réponse, pour chaque séquence."""
    logits = model(input_ids=ids, attention_mask=att).logits[:, :-1, :]
    targets = lab[:, 1:]
    # cross_entropy = -log p ; ignore_index masque le prompt et le padding
    token_logp = -F.cross_entropy(logits.transpose(1, 2), targets, ignore_index=-100, reduction="none")
    return token_logp.sum(dim=-1)

def dpo_forward(model, batch_pairs, beta):
    """Renvoie (perte DPO, marge implicite moyenne, accuracy implicite, logp_policy[chosen/rejected])."""
    ids, att, lab = make_batch(batch_pairs)
    B = len(batch_pairs)
    # log-probas de la POLITIQUE (adaptateur LoRA actif)
    logp_pol = sequence_logps(model, ids, att, lab)
    # log-probas de la RÉFÉRENCE (adaptateur désactivé, sans gradient)
    with torch.no_grad(), model.disable_adapter():
        logp_ref = sequence_logps(model, ids, att, lab)
    pol_c, pol_r = logp_pol[:B], logp_pol[B:]
    ref_c, ref_r = logp_ref[:B], logp_ref[B:]
    margin = beta * ((pol_c - ref_c) - (pol_r - ref_r))      # différence des récompenses implicites
    loss = -F.logsigmoid(margin).mean()
    return loss, margin.detach(), pol_c.detach(), pol_r.detach()

# Vérification : au départ, la perte DPO doit valoir ≈ log 2 = 0.693
policy.eval()
with torch.no_grad():
    l0, m0, _, _ = dpo_forward(policy, dpo_train_pairs[:4], beta=0.1)
print(f"Perte DPO initiale = {l0.item():.4f} (attendu ≈ {math.log(2):.4f}) | marge moyenne = {m0.mean().item():.4f}")

### Entraînement (nombre d'étapes limité)

Paramètres modestes pour rester rapide : `DPO_STEPS` mises à jour, `BATCH` paires par mise à jour, LoRA seulement.

In [ ]:
DPO_STEPS = 80 if DEVICE == "cuda" else 15     # réduire si CPU
BATCH = 4
BETA_DPO = 0.1
LR_DPO = 1e-4

optimizer = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=LR_DPO)
order = list(range(len(dpo_train_pairs)))
random.Random(SEED).shuffle(order)

dpo_log = []
policy.train()
for step in range(DPO_STEPS):
    idx = [order[(step * BATCH + k) % len(order)] for k in range(BATCH)]
    batch = [dpo_train_pairs[i] for i in idx]
    loss, margin, _, _ = dpo_forward(policy, batch, BETA_DPO)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step(); optimizer.zero_grad()
    dpo_log.append({"step": step + 1, "loss": loss.item(),
                    "margin": margin.mean().item(), "acc": (margin > 0).float().mean().item()})
    if (step + 1) % 10 == 0:
        w = dpo_log[-10:]
        print(f"step {step+1:3d}/{DPO_STEPS} | loss {np.mean([x['loss'] for x in w]):.4f} | "
              f"marge {np.mean([x['margin'] for x in w]):+.3f} | acc implicite {np.mean([x['acc'] for x in w]):.2f}")

log_df = pd.DataFrame(dpo_log)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(log_df["step"], log_df["loss"].rolling(5, min_periods=1).mean()); ax[0].set_title("Perte DPO"); ax[0].set_xlabel("étape")
ax[1].plot(log_df["step"], log_df["margin"].rolling(5, min_periods=1).mean()); ax[1].set_title("Marge implicite (chosen − rejected)"); ax[1].set_xlabel("étape")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4.14 Comparaison du modèle initial et du modèle ajusté

On soumet les **mêmes prompts** aux deux modèles (génération déterministe, `do_sample=False`, pour une comparaison équitable) :
- modèle **initial** = LoRA désactivé ;
- modèle **ajusté** = LoRA activé.

On utilise ici des prompts **non vus** à l'entraînement (jeu `heldout_prompts`) et quelques prompts écrits à la main.

In [ ]:
@torch.no_grad()
def generate(prompt, use_adapter, max_new_tokens=120):
    policy.eval()
    inputs = tok(chat_prompt(prompt), return_tensors="pt", add_special_tokens=False).to(DEVICE)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id,
                  repetition_penalty=1.05)
    if use_adapter:
        out = policy.generate(**inputs, **kwargs)
    else:
        with policy.disable_adapter():
            out = policy.generate(**inputs, **kwargs)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

handwritten = ["What is the capital of Tunisia?",
               "Explain overfitting in two sentences.",
               "Will the stock market go up next week?",
               "Give me a tip to learn Python quickly."]
compare_prompts = handwritten + heldout_prompts[:4]

comparison = []
for pr in compare_prompts:
    comparison.append({"prompt": pr, "initial": generate(pr, False), "aligné": generate(pr, True)})

for c in comparison[:6]:
    print("=" * 90)
    print("PROMPT  :", textwrap.shorten(c["prompt"], 200))
    print("INITIAL :", textwrap.shorten(c["initial"], 500))
    print("ALIGNÉ  :", textwrap.shorten(c["aligné"], 500))

**Comment comparer (grille)** : pertinence (répond-il à la question ?), précision (faits corrects ?), concision (longueur/remplissage), respect des consignes (format demandé, « en deux phrases »…). Dans la Partie 5, on ajoute des mesures **chiffrées**.

> Avec seulement quelques dizaines d'étapes LoRA, les changements sont généralement **subtils** (style, longueur, structure) plutôt que spectaculaires. C'est normal et attendu dans ce TP.

## 4.15 Pourquoi DPO évite-t-il le reward model séparé et la boucle PPO ?

**Idée clé** : le problème RLHF régularisé par KL,

$$\max_\pi\ \mathbb{E}_{y\sim\pi}[r(x,y)] - \beta\,\mathrm{KL}(\pi\,\|\,\pi_{ref}),$$

a une **solution optimale en forme fermée** :

$$\pi^*(y|x) = \frac{1}{Z(x)}\,\pi_{ref}(y|x)\,\exp\!\Big(\frac{r(x,y)}{\beta}\Big).$$

En inversant cette relation, la **récompense s'exprime avec la politique** :

$$r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta\log Z(x).$$

Dans la probabilité de Bradley–Terry, ce qui compte est $r(x,y_w) - r(x,y_l)$ : le terme $\beta\log Z(x)$ **s'annule** (même prompt !). On peut donc écrire directement la vraisemblance des préférences **en fonction de la politique**, ce qui donne la perte DPO vue plus haut.

**Conséquences pratiques**
| | RLHF / PPO | DPO |
|---|---|---|
| Reward model séparé | oui (à entraîner puis à geler) | **non** (implicite) |
| Génération en ligne pendant l'entraînement | oui (très coûteux) | **non** : données de préférences fixes |
| Modèles en mémoire | politique + critique + référence + reward | politique + référence |
| Nature de l'entraînement | RL instable, nombreux hyperparamètres | **perte supervisée** simple et stable |

Le reward model n'a pas disparu : il est **implicite** dans $\beta\log\frac{\pi_\theta}{\pi_{ref}}$.

# Partie 5 — Évaluation comparative et réflexion

## 5.16 Comparaison sur des prompts non utilisés à l'entraînement

On mesure, pour le modèle initial et le modèle aligné, sur les `heldout_prompts` (jamais vus) :
- le **score du reward model** (Partie 2) ;
- la **longueur** des réponses (en mots) ;
- le **taux de victoire** : % de prompts où le modèle aligné reçoit un meilleur score que l'initial.

> ⚠️ Le score du reward model n'est **qu'un proxy** (cf. 2.7). On le complète donc par une lecture qualitative des réponses.

In [ ]:
N_EVAL = min(len(heldout_prompts), 20)
eval_prompts = heldout_prompts[:N_EVAL] + handwritten

init_resp = [generate(p, False) for p in eval_prompts]
align_resp = [generate(p, True) for p in eval_prompts]

s_init = score_texts(eval_prompts, init_resp)
s_align = score_texts(eval_prompts, align_resp)
len_init = [len(r.split()) for r in init_resp]
len_align = [len(r.split()) for r in align_resp]

eval_df = pd.DataFrame({"prompt": [textwrap.shorten(p, 60) for p in eval_prompts],
                        "RM initial": s_init, "RM aligné": s_align,
                        "mots initial": len_init, "mots aligné": len_align})
win_rate = np.mean(np.array(s_align) > np.array(s_init))
summary = pd.DataFrame({
    "Modèle": ["Initial", "Aligné (DPO+LoRA)"],
    "Score RM moyen": [np.mean(s_init), np.mean(s_align)],
    "Longueur moyenne (mots)": [np.mean(len_init), np.mean(len_align)],
})
print(f"Taux de victoire du modèle aligné selon le reward model : {win_rate:.0%} ({N_EVAL + len(handwritten)} prompts)")
display(summary.round(3))
eval_df.round(2).head(10)

## 5.17 Les préférences apprises en entraînement se retrouvent-elles en validation ?

Deux vérifications :
1. **Reward model** : précision de classement train vs validation (déjà calculée).
2. **DPO** : sur les paires train et validation, le modèle favorise-t-il `chosen` ? On mesure
   - **accuracy de vraisemblance** : $\log\pi(y_w|x) > \log\pi(y_l|x)$ (pour le modèle initial **et** l'aligné) ;
   - **marge implicite** et accuracy implicite (par rapport à la référence).

Un grand écart train → validation signale un **sur-apprentissage**.

In [ ]:
@torch.no_grad()
def dpo_metrics(pairs_subset, beta=BETA_DPO, bs=4):
    """Accuracy de vraisemblance (initial / aligné), marge implicite, KL estimée sur les chosen."""
    policy.eval()
    acc_pol, acc_ref, margins, kl_chosen = [], [], [], []
    for i in range(0, len(pairs_subset), bs):
        b = pairs_subset[i:i+bs]
        ids, att, lab = make_batch(b); B = len(b)
        lp = sequence_logps(policy, ids, att, lab)
        with policy.disable_adapter():
            lr_ = sequence_logps(policy, ids, att, lab)
        acc_pol += (lp[:B] > lp[B:]).float().tolist()
        acc_ref += (lr_[:B] > lr_[B:]).float().tolist()
        margins += (beta * ((lp[:B] - lr_[:B]) - (lp[B:] - lr_[B:]))).tolist()
        n_tok = (lab[:B, 1:] != -100).sum(-1).clamp(min=1)
        kl_chosen += ((lp[:B] - lr_[:B]) / n_tok).tolist()   # log-ratio moyen par token (dérive vs référence)
    return {"acc vraisemblance (initial)": np.mean(acc_ref), "acc vraisemblance (aligné)": np.mean(acc_pol),
            "marge implicite moy.": np.mean(margins), "acc implicite (marge>0)": np.mean(np.array(margins) > 0),
            "dérive log-ratio/token (chosen)": np.mean(kl_chosen)}

sub_train = dpo_train_pairs[:min(60, len(dpo_train_pairs))]
sub_val = dpo_val_pairs[:min(60, len(dpo_val_pairs))]
res_table = pd.DataFrame({"train (DPO)": dpo_metrics(sub_train), "validation (DPO)": dpo_metrics(sub_val)})
display(res_table.round(3))
print(f"\nReward model — accuracy de classement : train {RM_TRAIN_ACC:.3f} | validation {RM_VAL_ACC:.3f}")

**Interprétation** : si l'accuracy de vraisemblance du modèle **aligné** dépasse celle du modèle **initial** à la fois en train **et** en validation, le comportement préféré est bien **généralisé**. Si le gain est fort en train mais faible/nul en validation, le modèle a **mémorisé** les paires.

## 5.18 Risques : sur-apprentissage, biais d'annotation, sur-optimisation

| Risque | Description | Comment le repérer / limiter |
|---|---|---|
| **Sur-apprentissage** | avec peu de paires, le modèle mémorise des réponses plutôt que d'apprendre une préférence générale | écart train/validation ; peu d'époques, LoRA de petit rang, early stopping, plus de données |
| **Biais des annotations** | les préférences encodent les goûts des annotateurs (longueur, politesse, ton assuré) ; ici *chosen* vient d'un modèle plus fort, d'où un biais de **style** (réponses plus longues et plus structurées) | corrélation longueur/score (2.7), diversifier les annotateurs, normaliser la longueur |
| **Sur-optimisation du signal** (*reward hacking*, loi de Goodhart) | la politique exploite les défauts du reward model (PPO) ou de la marge implicite (DPO) : le score monte, la qualité réelle non | pénalité KL / β, arrêt précoce, évaluation indépendante (humains, benchmarks), ensembles de reward models |
| **Dérive vs référence** | le modèle s'éloigne du comportement initial et peut « oublier » des capacités | mesurer la dérive (log-ratio), β plus grand, LoRA peu intrusif |
| **DPO : baisse de vraisemblance du chosen** | DPO peut réduire à la fois la proba de *chosen* et de *rejected* tant que la marge augmente | suivre `log π(chosen)` en absolu, régularisation (ex. terme SFT sur chosen) |

## 5.19 Conclusion argumentée : DPO ou pipeline RLHF classique ?

### DPO est préférable quand…
- **Ressources limitées** (comme dans ce TP) : pas de génération en ligne, pas de critique, pas de reward model à entraîner — seulement politique + référence.
- On dispose d'un **jeu de préférences statique** de bonne qualité (hors-ligne).
- On veut un entraînement **simple, stable et reproductible** (perte supervisée, peu d'hyperparamètres).
- Le but est un **ajustement de style / comportement** relativement local par rapport au modèle initial.

### Le pipeline RLHF classique (reward model + PPO) est préférable quand…
- On veut **explorer en ligne** : la politique génère de nouvelles réponses, évaluées par le reward model, ce qui permet de dépasser les réponses présentes dans le jeu de préférences.
- Le **reward model est réutilisable** : filtrage de sorties (*best-of-n*), évaluation automatique, surveillance, ou réutilisation sur plusieurs politiques.
- La tâche a un **signal de récompense vérifiable ou programmable** (code qui passe les tests, preuve mathématique) : on remplace alors le reward model appris par une récompense fiable.
- On dispose de **beaucoup de calcul** et d'ingénierie pour stabiliser PPO, et on veut un **contrôle fin** (récompenses multiples, contraintes de sécurité).

### Synthèse
| Critère | DPO | RLHF/PPO |
|---|---|---|
| Simplicité et stabilité | ✅ | ❌ |
| Coût de calcul | ✅ faible | ❌ élevé |
| Exploration en ligne | ❌ (données fixes) | ✅ |
| Reward model explicite réutilisable | ❌ | ✅ |
| Risque de reward hacking | présent (implicite) | présent (explicite, observable) |

Dans les deux cas, la qualité finale dépend d'abord de la **qualité et de la diversité des préférences** : aucun algorithme ne corrige des annotations biaisées. Le plus sûr est de **combiner** un entraînement par préférences avec une **évaluation indépendante** (humaine et par benchmarks), de **surveiller la dérive** par rapport à la référence, et de **ne jamais prendre le score d'un reward model pour la vérité**.